# 03b — M1 MLP rerun (Ravi Sharma)

Reruns M1's training and evaluation **unchanged** on the saved
`fmc-m1-features` (same `X.npy`, `y.npy`, `subset.csv`, seed 42, CPU as in
the original run) so that `m1_results.json` is written by the script from
the run's own variables. The original file was typed in by hand.

The training cell is the original cell 9 of `03-m1-mlp.ipynb` verbatim. The
only addition is tracking `best_ep` and `stop_ep`, which consumes no random
numbers, so the run should reproduce the original exactly. If it does not,
that is reported as a finding, not patched over.

In [ ]:
import os, glob, json
import numpy as np, pandas as pd

def find_input(name, marker):
    """Inputs mount at /kaggle/input/datasets/<user>/<name>/; fall back to a
    search so a mount-layout change gives a clear message."""
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{marker}', recursive=True)
    print(f'{name}: expected path missing, search found {hits}')
    assert hits, f'{marker} not found under /kaggle/input'
    return os.path.dirname(hits[0])

F_DIR = find_input('fmc-m1-features', 'X.npy')
print(F_DIR, '->', sorted(os.listdir(F_DIR)))

# The same three arrays the original run held in memory when it trained:
# X and y were saved from those variables, and subset.csv from the same `sub`.
X   = np.load(f'{F_DIR}/X.npy')
y   = np.load(f'{F_DIR}/y.npy')
sub = pd.read_csv(f'{F_DIR}/subset.csv', dtype={'ytid': str})
assert X.shape == (10697, 86) and len(y) == len(sub) == len(X)
assert (sub.label.values == y).all()
print(X.dtype, sub.groupby(['split', 'label']).size().to_dict())

# The hand-typed original, kept for the reproduction check at the end.
orig = json.load(open(f'{F_DIR}/m1_results.json'))
print('original (hand-typed):', orig)

In [ ]:
# Original cell 9, verbatim except for best_ep / stop_ep bookkeeping.
import numpy as np, torch, torch.nn as nn, time
from sklearn.preprocessing import StandardScaler

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # kernel has no GPU: cpu, as originally

idx = {s: (sub.split == s).values for s in ['train','val','test']}

# Scaler is fit on train only. Fitting on all data would leak test statistics.
sc = StandardScaler().fit(X[idx['train']])
Xs = {s: torch.tensor(sc.transform(X[idx[s]]), dtype=torch.float32, device=dev)
      for s in idx}
ys = {s: torch.tensor(y[idx[s]], dtype=torch.float32, device=dev) for s in idx}

model = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 1)
).to(dev)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()

best_val, best_state, patience, bad = 0.0, None, 15, 0
best_ep, stop_ep = -1, None
t0 = time.time()
for ep in range(200):
    model.train()
    perm = torch.randperm(len(Xs['train']), device=dev)
    for i in range(0, len(perm), 128):
        b = perm[i:i+128]
        opt.zero_grad()
        loss = lossf(model(Xs['train'][b]).squeeze(1), ys['train'][b])
        loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        # Selection on PLAIN validation accuracy (locked decision 10).
        va = ((model(Xs['val']).squeeze(1) > 0).float() == ys['val']).float().mean().item()
    if va > best_val:
        best_val, bad, best_ep = va, 0, ep
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            stop_ep = ep
            print('early stop at epoch', ep); break

model.load_state_dict(best_state)
train_time = time.time() - t0
print(f'best val acc: {best_val:.4f} at epoch {best_ep} | train time: {train_time:.1f}s')

In [ ]:
# Original cell 10 metric code, then the results file written FROM VARIABLES.
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

model.eval()
with torch.no_grad():
    logits = model(Xs['test']).squeeze(1)
    score  = torch.sigmoid(logits).cpu().numpy()
pred = (score > 0.5).astype(int)
true = y[idx['test']]

fpr, tpr, _ = roc_curve(true, score)
fnr = 1 - tpr
eer = float(fpr[np.nanargmin(np.abs(fnr - fpr))])   # where FAR and FRR cross

m1 = {
    "balanced_accuracy": balanced_accuracy_score(true, pred),
    "precision": precision_score(true, pred),
    "recall":    recall_score(true, pred),
    "f1":        f1_score(true, pred),
    "auc":       roc_auc_score(true, score),
    "eer":       eer,
}
for k, v in m1.items():
    print(f"{k:20s} {v:.4f}")

results = {"model": "M1_MLP", "seed": SEED, "device": dev,
           "train_time_s": round(train_time, 1),
           "best_val_acc": round(best_val, 4), "best_epoch": best_ep,
           "early_stop_epoch": stop_ep, "selection_metric": "val_accuracy",
           "params": sum(p.numel() for p in model.parameters()),
           **{k: round(float(v), 4) for k, v in m1.items()},
           "n_test": int(len(true)),
           "torch_version": torch.__version__,
           "provenance": "written by script from run variables (03b rerun); "
                         "replaces the hand-typed file from 03-m1-mlp"}
json.dump(results, open('/kaggle/working/m1_results.json', 'w'), indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Reproduction check against the hand-typed original. Train time is wall-clock
# and is not expected to match; everything else should, to 4 dp.
keys = ['best_val_acc', 'early_stop_epoch', 'balanced_accuracy', 'precision',
        'recall', 'f1', 'auc', 'eer']
check = {k: {'original': orig.get(k), 'rerun': results[k],
             'match': orig.get(k) == results[k]} for k in keys}
print(pd.DataFrame(check).T.to_string())
json.dump({"all_match": all(v['match'] for v in check.values()), "fields": check},
          open('/kaggle/working/m1_reproduction_check.json', 'w'), indent=2)
print('ALL MATCH' if all(v['match'] for v in check.values()) else 'MISMATCH: report this')